# Lab 1: Clasificacion de Imagenes con KerasHub y Keras 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-source-models/blob/main/session-01-hf-kerashub-litert/01-kerashub-image-classification/03_imagenet_classification_basics.ipynb)

## Objetivo
Aprender a cargar, inspeccionar y ejecutar modelos pre-entrenados de vision por computador utilizando el ecosistema **KerasHub** y la flexibilidad multi-backend de **Keras 3** sobre PyTorch.

### Paso 1: Instalacion de Dependencias

In [ ]:
!pip install -q --upgrade keras keras-hub torch torchvision pillow matplotlib requests

### Paso 2: Configuracion del Backend y Certificados SSL
Keras 3 permite desacoplar el frontend del motor de computo (PyTorch, JAX, TensorFlow).

In [ ]:
import os
import ssl
import urllib.request

# Asegurar certificados SSL
ssl._create_default_https_context = ssl._create_unverified_context
os.environ["KERAS_BACKEND"] = "torch"

import keras
import keras_hub
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import time

print(f"Keras Version: {keras.__version__}")
print(f"Active Backend: {keras.config.backend()}")

### Paso 3: Descarga de Imagenes de Prueba (Perro Golden Retriever, Gato, Auto y Grace Hopper)

In [ ]:
os.makedirs("sample_images", exist_ok=True)

sample_urls = {
    "dog.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b3/Golden_Retriever_2019.jpg/500px-Golden_Retriever_2019.jpg",
    "cat.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/500px-Cat_November_2010-1a.jpg",
    "car.jpg": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/2019_Toyota_Corolla_Icon_Tech_VVT-i_Hybrid_1.8.jpg/500px-2019_Toyota_Corolla_Icon_Tech_VVT-i_Hybrid_1.8.jpg",
    "grace_hopper.jpg": "https://storage.googleapis.com/download.tensorflow.org/example_images/grace_hopper.jpg"
}

headers = {"User-Agent": "Mozilla/5.0"}
ctx = ssl._create_unverified_context()

for name, url in sample_urls.items():
    path = os.path.join("sample_images", name)
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, context=ctx) as resp, open(path, "wb") as f:
        f.write(resp.read())
    print(f"Descargada con exito: {name}")

### Paso 4: Carga del Modelo Clasificador de Vision
Cargamos MobileNetV3 con pesos pre-entrenados en ImageNet (1,000 clases).

In [ ]:
from keras.applications import MobileNetV3Small

model = MobileNetV3Small(weights="imagenet")
print(f"Modelo MobileNetV3 cargado. Parametros totales: {model.count_params():,}")

### Paso 5: Funcion de Inferencia y Decodificacion de Clases

In [ ]:
def classify_image(image_path, top_k=5):
    img = Image.open(image_path).convert("RGB")
    img_resized = img.resize((224, 224))
    arr = np.array(img_resized, dtype=np.float32)
    tensor = np.expand_dims(arr, axis=0)
    
    start_time = time.time()
    predictions = model.predict(tensor, verbose=0)
    elapsed_ms = (time.time() - start_time) * 1000
    
    from keras.applications.mobilenet_v3 import decode_predictions
    results = decode_predictions(predictions, top=top_k)[0]
    
    plt.figure(figsize=(5, 3))
    plt.imshow(img)
    plt.axis("off")
    title_text = f"Top: {results[0][1]} ({results[0][2]*100:.1f}%)\nLatencia: {elapsed_ms:.1f}ms"
    plt.title(title_text)
    plt.show()
    
    print(f"Resultados para {os.path.basename(image_path)}:")
    for rank, (class_id, label, score) in enumerate(results, start=1):
        print(f"  {rank}. {label:<25} {score*100:.2f}%")
    print("-" * 40)
    return results

### Paso 6: Ejecucion de Inferencias sobre Imagenes Reales

In [ ]:
for img_name in ["dog.jpg", "cat.jpg", "car.jpg", "grace_hopper.jpg"]:
    path = os.path.join("sample_images", img_name)
    classify_image(path, top_k=3)

### Paso Final: Limpieza de Memoria y Archivos Descargados

In [ ]:
import gc, shutil
import torch

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if os.path.exists('sample_images'):
    shutil.rmtree('sample_images')
print('Archivos temporales e imagenes eliminados correctamente.')